# nanoql: first steps

nanoql prices an instrument as a plain function of its inputs: the instrument, an evaluation context that carries the date, and the market data its engine takes. This notebook runs in Google Colab, or locally after `pip install nanoql jupyter`. Its option cell follows QuantLib's EquityOption example.

The first cell installs the library.

In [ ]:
%pip install "nanoql>=2.0,<3"

## A first swap

A five-year SOFR swap on a flat curve. The curve, the swap and the evaluation date are values, and the engine is a function of them.

In [ ]:
import datetime

from nanoql import ConventionSet, DiscountingSwapEngine, EvaluationContext, FlatForward, MakeOIS, Period, TimeUnit

today = datetime.date(2026, 10, 15)
context = EvaluationContext(today)
qs = ConventionSet.quant_lib()
curve = FlatForward(today, 0.036, qs.day_counter("Actual365Fixed"))

swap = (MakeOIS(Period(5, TimeUnit.Years), qs.overnight_index("Sofr", curve))
        .with_fixed_rate(0.036).with_nominal(10e6)
        .to_overnight_indexed_swap(context))

result = DiscountingSwapEngine.calculate(swap, context, curve)
print(f"NPV {result.npv:,.2f}, fair rate {result.fair_rate:.4%}")

## A SOFR curve from swap quotes

A curve bootstrapped from six OIS quotes, its discount factors at the quotes' tenors:

In [ ]:
from nanoql import OISRateHelper, PiecewiseLogLinearDiscount

quotes = {1: 0.0372, 2: 0.0358, 3: 0.0352, 5: 0.0350, 7: 0.0353, 10: 0.0360}
sofr = qs.overnight_index("Sofr")
helpers = [OISRateHelper(2, Period(years, TimeUnit.Years), rate, sofr, context,
                         forward_start=Period(0, TimeUnit.Days))  # spot-starting swaps
           for years, rate in quotes.items()]
sofr_curve = PiecewiseLogLinearDiscount(today, helpers, qs.day_counter("Actual365Fixed"), context)

for years, rate in quotes.items():
    print(f"{years:>2}Y  discount {sofr_curve.discount(float(years)):.6f}  quote {rate:.4%}")

The same swap priced on the bootstrapped curve:

In [ ]:
swap = (MakeOIS(Period(5, TimeUnit.Years), qs.overnight_index("Sofr", sofr_curve))
        .with_fixed_rate(0.036).with_nominal(10e6)
        .to_overnight_indexed_swap(context))
result = DiscountingSwapEngine.calculate(swap, context, sofr_curve)
print(f"NPV {result.npv:,.2f}, fair rate {result.fair_rate:.4%}")

## An American put

QuantLib's EquityOption example's put, struck at 40 on a spot of 36, with a 6% rate, no dividends, 20% volatility and a year to expiry: European by Black-Scholes, and American by Barone-Adesi and Whaley's approximation.

In [ ]:
from nanoql import (AmericanExercise, AnalyticEuropeanEngine, BaroneAdesiWhaleyApproximationEngine, BlackConstantVol,
                    BlackScholesMertonProcess, EuropeanExercise, OptionType, PlainVanillaPayoff, VanillaOption)

trade_date = datetime.date(1998, 5, 15)
settlement = datetime.date(1998, 5, 17)
maturity = datetime.date(1999, 5, 17)
option_context = EvaluationContext(trade_date)
act365 = qs.day_counter("Actual365Fixed")
process = BlackScholesMertonProcess(
    36.0,
    FlatForward(settlement, 0.0, act365),
    FlatForward(settlement, 0.06, act365),
    BlackConstantVol(settlement, qs.calendar("TARGET"), 0.20, act365))

payoff = PlainVanillaPayoff(OptionType.Put, 40.0)
european = VanillaOption(payoff, EuropeanExercise(maturity))
american = VanillaOption(payoff, AmericanExercise(settlement, maturity))

print(f"European, Black-Scholes:       {AnalyticEuropeanEngine.calculate(european, option_context, process).npv:.6f}")
print(f"American, Barone-Adesi-Whaley: "
      f"{BaroneAdesiWhaleyApproximationEngine.calculate(american, option_context, process).npv:.6f}")

## A default curve from CDS spreads

A hazard-rate curve bootstrapped from one-, three- and five-year CDS spreads, and its survival probabilities:

In [ ]:
from nanoql import (BusinessDayConvention, CreditDefaultSwap, DateGenerationRule, Frequency, MakeSchedule,
                    MidPointCdsEngine, PiecewiseFlatHazardRate, ProtectionSide, SpreadCdsHelper)

target = qs.calendar("TARGET")
act360 = qs.day_counter("Actual360")
recovery = 0.4
spreads = {1: 0.0080, 3: 0.0110, 5: 0.0135}
cds_helpers = [
    SpreadCdsHelper(spread, Period(years, TimeUnit.Years), 0, target, Frequency.Quarterly,
                    BusinessDayConvention.Following, DateGenerationRule.TwentiethIMM, act360, recovery, curve, context)
    for years, spread in spreads.items()]
hazard_curve = PiecewiseFlatHazardRate(today, cds_helpers, act365, context)

for years in spreads:
    print(f"{years}Y survival probability {hazard_curve.survival_probability(float(years)):.4%}")

A five-year CDS at the quoted spread, priced on that curve:

In [ ]:
maturity = target.adjust(today.replace(year=today.year + 5), BusinessDayConvention.Following)
schedule = (MakeSchedule().from_(today).to(maturity)
            .with_frequency(Frequency.Quarterly).with_calendar(target)
            .with_termination_date_convention(BusinessDayConvention.Unadjusted)
            .with_rule(DateGenerationRule.TwentiethIMM)
            .to_schedule())
cds = CreditDefaultSwap(ProtectionSide.Buyer, 10e6, 0.0135, schedule, BusinessDayConvention.Following, act360)
cds_result = MidPointCdsEngine.calculate(cds, context, hazard_curve, recovery, curve)
# The NPV is round-off, its sign the platform's last bits; rounded and added to 0.0, it prints as 0.00.
print(f"NPV {round(cds_result.npv, 2) + 0.0:,.2f}, fair spread {cds_result.fair_spread:.4%}")

## Next

The [documentation](https://nanoql.org/docs/) has the guides, the API reference and the bindings' rules.